# Soft Robotic Arm — Pouch Control and Circular Tracking

This notebook teaches you how to:

1. command all twenty pouches independently;
2. read the five pouch pressures in every segment;
3. observe the arm tip position and velocity;
4. use a PD controller to track a circular tip trajectory; and
5. render the controller's motion as a simulation video.

The final example is a starting point: tune the gains and pouch distribution,
then improve or replace the controller.


## 1 — Install the arm library

Run this once in a new Google Colab runtime. It installs `soft-robotic-arm`, a
ready-to-use MuJoCo simulation of the four-segment pneumatic arm. The package
includes the configured robot model and its calibration data, so you do not
need to download the repository, open the model files, or build the robot
yourself. MuJoCo, NumPy, Matplotlib, and the video tools are installed with it.

After installation, you use the package through a small public interface:

```python
from soft_robotic_arm import make_sim

sim = make_sim(control_hz=100, seed=0)
obs = sim.reset()
obs = sim.step(command)
```

- `make_sim(...)` creates a correctly configured arm simulation.
- `reset()` returns the arm to its initial state and gives you its first sensor reading.
- `step(command)` applies one pressure command and advances the simulation by one
  control interval.
- `command` may contain four segment pressures with shape `(4,)`, or twenty
  individual pouch pressures with shape `(4, 5)`.
- The returned `obs` dictionary contains the tip position and velocity, simulation
  time, and measured segment and pouch pressures.

The install command downloads the published package and all required dependencies
directly from PyPI. If an older version was already loaded in Colab, the setup
cell reloads the installed package before creating the simulator.


In [ ]:
%pip install soft-robotic-arm


## 2 — Segment and pouch indexing

A pouch command has shape `(4, 5)`. Rows select S1–S4 and columns select
P1–P5. For example, `command[1, 2]` addresses P3 in S2.

<img src="https://raw.githubusercontent.com/Jeevan-HM/Soft-Robotic-Arm/simulation/mujoco/coursework/assets/soft_arm_hardware_and_layout.png" alt="Soft robotic arm hardware beside its segment and pouch indexing diagram" width="100%"/>

*Physical arm and indexing diagram. P1 is the bottom pouch and P5 is the top pouch in each segment.*

| Row | Segment | Tip tendency |
|---:|---|---|
| `0` | S1 | +x |
| `1` | S2 | +y |
| `2` | S3 | −x |
| `3` | S4 | −y |


In [ ]:
import os
import sys
from importlib.metadata import version

# Google Colab is Linux. Set the renderer before importing MuJoCo.
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

# Colab can retain an already-imported package after %pip upgrades it.
# Remove only this package from the import cache, then load the installed copy.
for module_name in tuple(sys.modules):
    if module_name == "soft_robotic_arm" or module_name.startswith("soft_robotic_arm."):
        del sys.modules[module_name]

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import soft_robotic_arm
from IPython.display import Video, display
from soft_robotic_arm import make_sim

installed_version = version("soft-robotic-arm")
CONTROL_HZ = 100.0
MAX_PRESSURE_PSI = 9.0

sim = make_sim(control_hz=CONTROL_HZ, seed=0)
obs = sim.reset()

print(f"soft-robotic-arm version: {installed_version}")
print("Simulator ready")
print("Command rows: S1, S2, S3, S4; columns: P1, P2, P3, P4, P5")
print("Tip position [m]:", np.round(obs["tip_pos"], 4))
print("Pouch-pressure array shape:", obs["pouch_pressures"].shape)


### What the simulator setup code does

- Selects EGL so MuJoCo can render without a physical display in Colab.
- Clears any older `soft_robotic_arm` import left in the current runtime.
- Imports NumPy, Matplotlib, ImageIO, and the `make_sim` simulator factory.
- Reports the installed package version.
- Creates the arm at a 100 Hz control rate; `seed=0` makes sensor noise repeatable.
- Calls `sim.reset()` to place the arm in its initial state and return the first
  sensor observation in `obs`.

## 3 — Command every pouch and read its pressure

Use a `(4, 5)` array to command the twenty pouches independently:

```python
command[segment_index, pouch_index] = pressure_psi
obs = sim.step(command)
```

Rows are S1–S4; columns are P1–P5. `sim.step()` also accepts a `(4,)` segment
vector and broadcasts each segment value to its five pouches. Keep every
command between 0 and 9 psi.

| Observation | Shape | Meaning |
|---|---:|---|
| `obs["tip_pos"]` | `(3,)` | tip `[x, y, z]` position in metres |
| `obs["tip_vel"]` | `(3,)` | tip velocity in m/s |
| `obs["pouch_pressures"]` | `(4, 5)` | measured pressure of every pouch |
| `obs["segment_pressures"]` | `(4,)` | mean measured pressure of each segment |
| `obs["time"]` | scalar | simulation time in seconds |


In [ ]:
# Commands are absolute pressures in psi. Rows select S1--S4 and
# columns select P1--P5, so the complete command has shape (4, 5).
command = np.full((4, 5), 2.0)  # Hold every pouch at a 2 psi baseline.
command[1, 2] = 6.0             # Raise only S2, P3 to 6 psi.

# One call to step() advances one control interval (1 / CONTROL_HZ seconds).
# Repeating it CONTROL_HZ times therefore applies this command for one second.
for _ in range(int(CONTROL_HZ)):
    obs = sim.step(command)

# The commanded and measured values differ because the plant includes pneumatic
# lag, deformation-dependent sensing, and sensor noise.
print("Commanded pouch pressures [psi]:\n", command)
print("Measured pouch pressures [psi]:\n", np.round(obs["pouch_pressures"], 3))
print("Measured segment means [psi]:", np.round(obs["segment_pressures"], 3))
print("Tip position [m]:", np.round(obs["tip_pos"], 4))


### Try any pouch

Choose `TEST_SEGMENT` from 0–3 and `TEST_POUCH` from 0–4. The cell holds every
other pouch at 2 psi, raises only the selected pouch, and plots its response.


In [ ]:
# Change these two indices to investigate any of the twenty pouches.
TEST_SEGMENT = 0  # 0=S1, 1=S2, 2=S3, 3=S4
TEST_POUCH = 0    # 0=P1, 1=P2, 2=P3, 3=P4, 4=P5

# Use a fresh simulator so this experiment does not inherit the previous state.
sim = make_sim(control_hz=CONTROL_HZ, seed=1)
obs = sim.reset()
baseline = np.full((4, 5), 2.0)

# Let the pneumatic and mechanical states settle before measuring displacement.
for _ in range(int(3.0 * CONTROL_HZ)):
    obs = sim.step(baseline)
home = obs["tip_pos"].copy()  # Reference position after settling [m].

# Perturb one pouch while leaving the other nineteen at the baseline pressure.
command = baseline.copy()
command[TEST_SEGMENT, TEST_POUCH] = 6.0

# Record four seconds of tip motion and all twenty measured pressures.
times, tips, measured = [], [], []
for _ in range(int(4.0 * CONTROL_HZ)):
    obs = sim.step(command)
    times.append(obs["time"])
    tips.append(obs["tip_pos"].copy())
    measured.append(obs["pouch_pressures"].copy())

# Convert lists to arrays for vectorized analysis and plotting.
times = np.asarray(times) - times[0]  # Start this plot at t = 0 s.
tips = np.asarray(tips)               # Shape: (samples, xyz).
measured = np.asarray(measured)       # Shape: (samples, segment, pouch).

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# Plot planar displacement relative to the settled position; convert m to mm.
axes[0].plot(times, (tips[:, 0] - home[0]) * 1000, label="Δx")
axes[0].plot(times, (tips[:, 1] - home[1]) * 1000, label="Δy")
axes[0].set(xlabel="Time [s]", ylabel="Tip change [mm]", title="Tip response")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Show all five pressure sensors in the selected segment.
for pouch in range(5):
    axes[1].plot(times, measured[:, TEST_SEGMENT, pouch], label=f"P{pouch + 1}")
axes[1].set(xlabel="Time [s]", ylabel="Measured pressure [psi]", title=f"S{TEST_SEGMENT + 1} pouch pressures")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4 — Cartesian PD control with pressure allocation

This example is a closed-loop task-space controller. At each 100 Hz update it
uses the measured planar tip position and velocity, then follows this architecture:

`circular reference → Cartesian PD → opposing segment pressures → pouch allocation and saturation → arm physics → tip feedback`

For radius $R$ and angular speed $\omega$, the desired position and velocity are

$\mathbf r_d(t)=\mathbf c+R[\cos(\omega t),\sin(\omega t)]^T$

$\mathbf v_d(t)=R\omega[-\sin(\omega t),\cos(\omega t)]^T$.

The feedback law produces a planar **pressure correction**:

$\Delta\mathbf p_{xy}=K_p(\mathbf r_d-\mathbf r)+K_d(\mathbf v_d-\mathbf v)$.

Thus $K_p$ has units psi/m and $K_d$ has units psi·s/m. The controller applies
$+\Delta p_x$ to S1 and $-\Delta p_x$ to the opposing S3; similarly, it applies
$+\Delta p_y$ to S2 and $-\Delta p_y$ to S4. These are only pressure commands.
They do not prescribe the arm's position or bending: MuJoCo determines the
resulting motion from pressure-generated forces, stiffness, damping, gravity,
inertia, and pneumatic lag. A symmetric 4.5 psi bias provides room for both
signs without creating a lateral preference.

Finally, `pouch_profile` statically allocates each segment correction across its
five pouches:

$p_{s,k}=\operatorname{clip}(p_0+w_k\Delta p_s,0,p_{max})$.

All twenty pressures are therefore addressed explicitly, although the feedback
law has two controlled degrees of freedom, x and y—not twenty independent loops.
`K_p` sets correction strength, while `K_d` adds damping and reduces phase lag.
Saturation, pneumatic lag, and nonlinear arm mechanics limit performance, so this
is a defensible baseline controller rather than an inverse-dynamics controller.
On hardware, the pressure regulators would normally provide a faster inner loop.


In [ ]:
class CircularPDController:
    """Cartesian tip-space PD controller with static pouch allocation.

    Students can tune the constructor parameters, change reference(), or replace
    compute(). Keep the compute(t, obs) interface and return a finite (4, 5)
    array of absolute pouch pressures in psi.
    """

    def __init__(self, radius_m=0.015, period_s=10.0, kp=500.0, kd=20.0):
        # Reference-trajectory parameters.
        self.radius = radius_m
        self.omega = 2.0 * np.pi / period_s

        # Cartesian feedback gains. Tune these first when studying performance.
        self.kp = kp  # Position gain [psi/m].
        self.kd = kd  # Velocity gain [psi·s/m].

        # A mid-range bias lets the controller increase one segment's pressure
        # while decreasing the opposing segment without requesting negative psi.
        self.baseline_psi = 4.5
        self.center_xy = None  # Set from the measured reset position on first use.

        # Distribute each segment correction across its five pouch commands.
        # The mean is one, so this profile preserves the mean segment correction.
        self.pouch_profile = np.array([0.70, 0.85, 1.00, 1.15, 1.30])

    def reference(self, t):
        """Return desired planar tip position [m] and velocity [m/s]."""
        angle = self.omega * t
        position = self.center_xy + self.radius * np.array([np.cos(angle), np.sin(angle)])
        # Analytical derivative of the circular position reference.
        velocity = self.radius * self.omega * np.array([-np.sin(angle), np.cos(angle)])
        return position, velocity

    def compute(self, t: float, obs: dict) -> np.ndarray:
        """Compute one (segment, pouch) pressure command from tip feedback."""
        # Define the circle around the measured initial x-y tip position.
        if self.center_xy is None:
            self.center_xy = obs["tip_pos"][:2].copy()

        # Outer-loop task-space PD feedback. Only x and y are controlled here.
        target_pos, target_vel = self.reference(t)
        position_error = target_pos - obs["tip_pos"][:2]
        velocity_error = target_vel - obs["tip_vel"][:2]
        xy_pressure_correction = self.kp * position_error + self.kd * velocity_error

        # Form opposing pressure changes explicitly. S1/S3 are the x pair and
        # S2/S4 are the y pair. This selects pressures; the simulated mechanics
        # determine the bending and resulting tip motion.
        x_pressure_delta, y_pressure_delta = xy_pressure_correction
        segment_pressure_delta = np.array([
            x_pressure_delta,   # Increase S1 for a positive x correction.
            y_pressure_delta,   # Increase S2 for a positive y correction.
            -x_pressure_delta,  # Decrease the opposing S3 pressure.
            -y_pressure_delta,  # Decrease the opposing S4 pressure.
        ])

        # Apply a different share of each segment correction to all five pouches.
        command = (
            self.baseline_psi
            + segment_pressure_delta[:, None] * self.pouch_profile[None, :]
        )

        # Pressure saturation represents the allowed actuator command range.
        return np.clip(command, 0.0, MAX_PRESSURE_PSI)


## 5 — Track and plot the circle

The loop advances the simulator, records the desired and measured paths at the
same timestamps, and reports planar tracking RMSE. The controller must return a
finite `(4, 5)` pouch-pressure matrix on every call. The circle starts 15 mm
from the reset position, so the reported RMSE includes the initial acquisition
transient as well as steady tracking.


In [ ]:
# Create a fresh plant and controller for a reproducible closed-loop run.
sim = make_sim(control_hz=CONTROL_HZ, seed=2)
controller = CircularPDController()
obs = sim.reset()

RUN_TIME_S = 20.0
VIDEO_FPS = 10
VIDEO_PATH = "soft_arm_pd_tracking.mp4"
VIDEO_STRIDE = max(1, int(round(CONTROL_HZ / VIDEO_FPS)))
times, references, tips, commands = [], [], [], []

# Write rendered frames as the controller runs. The video is sampled more
# slowly than the 100 Hz controller so rendering does not dominate the run.
video_writer = imageio.get_writer(
    VIDEO_PATH, fps=VIDEO_FPS, codec="libx264", quality=7
)
print(f"Running {RUN_TIME_S:.0f} s simulation and recording {VIDEO_PATH} ...")

# Standard sampled-data loop: measure -> control -> actuate -> repeat.
for step_index in range(int(RUN_TIME_S * CONTROL_HZ)):
    # The controller uses only the current time, current observation, and its
    # internal state. It must not call sim.step() itself.
    command = np.asarray(controller.compute(obs["time"], obs), dtype=float)

    # Fail early if a student controller violates the required output contract.
    if command.shape != (4, 5) or not np.all(np.isfinite(command)):
        raise ValueError("Controller must return a finite (4, 5) pouch-pressure matrix")

    # Apply the command for one 100 Hz control interval.
    obs = sim.step(command)
    # Evaluate the reference at the new observation time for a fair comparison.
    target_pos, _ = controller.reference(obs["time"])

    # copy() prevents later simulator updates from changing stored samples.
    times.append(obs["time"])
    references.append(target_pos.copy())
    tips.append(obs["tip_pos"][:2].copy())
    commands.append(command.copy())

    # Capture the actual MuJoCo scene produced by this controller command.
    if step_index % VIDEO_STRIDE == 0:
        frame = sim.render_frame(width=640, height=480)
        video_writer.append_data(frame)

video_writer.close()
print(f"Video saved to {VIDEO_PATH}")

# Stack logged samples into arrays for metrics and plotting.
times = np.asarray(times)                    # (samples,)
references = np.asarray(references)          # (samples, xy)
tips = np.asarray(tips)                      # (samples, xy)
commands = np.asarray(commands)              # (samples, segment, pouch)

# Euclidean planar error at each sample and one scalar summary in millimetres.
errors = np.linalg.norm(references - tips, axis=1)
rmse_mm = 1000.0 * np.sqrt(np.mean(errors**2))
print(f"Planar tracking RMSE: {rmse_mm:.2f} mm")
print("Command history shape:", commands.shape, "= time × segment × pouch")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# Desired and measured paths in the x-y task space.
axes[0].plot(references[:, 0] * 1000, references[:, 1] * 1000, "--", label="reference")
axes[0].plot(tips[:, 0] * 1000, tips[:, 1] * 1000, label="tip")
axes[0].set(xlabel="x [mm]", ylabel="y [mm]", title="Circular tracking")
axes[0].set_aspect("equal")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Instantaneous tracking-error magnitude.
axes[1].plot(times, errors * 1000)
axes[1].set(xlabel="Time [s]", ylabel="Planar error [mm]", title="Tracking error")
axes[1].grid(True, alpha=0.3)

# Show the five commands allocated to S1; inspect other segments similarly.
for pouch in range(5):
    axes[2].plot(times, commands[:, 0, pouch], label=f"S1 P{pouch + 1}")
axes[2].set(xlabel="Time [s]", ylabel="Command [psi]", title="Independent S1 pouch commands")
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Controller simulation video

This is the rendered output of the controller run above—not a pre-recorded
clip. Rerun the experiment after changing the gains, reference, or pouch
allocation to generate a new video of that controller.


In [ ]:
# Embed the generated MP4 in the notebook and provide Colab's video controls.
display(Video(VIDEO_PATH, embed=True, width=800, html_attributes="controls loop"))
sim.close()  # Release MuJoCo's off-screen renderer after displaying the video.


### Student tasks

- Tune `kp` and `kd` and explain their effect on tracking.
- Change the circle radius or period and compare the RMSE.
- Change `pouch_profile` to investigate how P1–P5 affect the motion.
- Keep every command between 0 and 9 psi.
- Explain how the `(4, 5)` command maps to the twenty pouches.
